# Análisis inicial y resampling

Agregación **diaria × espacial** de los siniestros: grilla de celdas de 500 × 500 m sobre las
coordenadas UTM. Cada fila del conjunto resultante es una combinación **(día, zona)** y la
variable objetivo es la cantidad de siniestros en esa celda ese día.

Se excluyen `Gravedad`, `Tipo de Siniestro`, `fixed` y `Hora` (el resampling es diario), y las
coordenadas `X`/`Y` originales, ya que la ubicación queda representada por la zona.

El panel se completa con ceros para los días sin siniestros, **restringido a las celdas observadas**
(las que tienen al menos un siniestro histórico).

In [1]:
from pathlib import Path

import holidays
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq

In [2]:
# Ruta relativa a la raíz del repo: funciona sin importar desde dónde se abra el notebook
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir())
RUTA_CSV = RAIZ / "data" / "raw" / "uru_siniestros_unificado.csv"

# skipinitialspace: el CSV trae un espacio después de cada coma
df = pd.read_csv(RUTA_CSV, skipinitialspace=True, encoding="utf-8")
df.columns = df.columns.str.strip()  # los nombres de columna también traen espacios
df["Fecha"] = pd.to_datetime(df["Fecha"], format="%m/%d/%Y")

df.shape

(224694, 11)

In [3]:
df.head()

,Fecha,Calle,Tipo de Siniestro,Gravedad,Dia Semana,Hora,Departamento,Localidad,fixed,X,Y
0,2018-07-24,AVENIDA DIECIOCHO DE JULIO,ATROPELLO DE PEATÓN,LEVE,MARTES,16,MONTEVIDEO,MONTEVIDEO,1,575370,6137570
1,2018-05-14,CAMINO FELIPE CARDOSO,CAÍDA,LEVE,LUNES,11,MONTEVIDEO,MONTEVIDEO,1,581630,6141550
2,2018-10-08,NO SE INGRESO,CAÍDA,LEVE,LUNES,17,CANELONES,SIN DATOS,1,609595,6152040
3,2018-08-13,ATALA TOSCANO DE MENDIVEL,COLISIÓN ENTRE VEHÍCULOS,SIN LESIONADOS,LUNES,20,COLONIA,OMBUES DE LAVALLE,1,425305,6244210
4,2018-10-15,ROSA SPRINGER DE KEEL,COLISIÓN ENTRE VEHÍCULOS,SIN LESIONADOS,LUNES,5,SAN JOSE,ECILDA PAULLIER,1,494550,6197445


## 1. Grilla espacial y catálogo de zonas

In [4]:
TAM_CELDA = 1000  # metros

df["celda_x"] = (df["X"] // TAM_CELDA).astype("int32")
df["celda_y"] = (df["Y"] // TAM_CELDA).astype("int32")

# Catálogo de zonas observadas: sólo las celdas con al menos un siniestro histórico.
# zona_id va de 0 a n-1 para poder usarlo directamente como índice de columna más abajo.
zonas = (
    df.groupby(["celda_x", "celda_y"], as_index=False)
    .agg(n_hist=("Fecha", "size"), departamento=("Departamento", lambda s: s.mode().iat[0]))
    .sort_values(["celda_x", "celda_y"])
    .reset_index(drop=True)
)
zonas.insert(0, "zona_id", np.arange(len(zonas), dtype="int32"))
# Centroide de la celda, para mapear después las zonas de mayor riesgo
zonas["centro_x"] = zonas["celda_x"] * TAM_CELDA + TAM_CELDA // 2
zonas["centro_y"] = zonas["celda_y"] * TAM_CELDA + TAM_CELDA // 2

df = df.merge(zonas[["celda_x", "celda_y", "zona_id"]], on=["celda_x", "celda_y"], how="left")
assert df["zona_id"].notna().all(), "hay siniestros sin zona asignada"

print(f"zonas observadas: {len(zonas):,}")
zonas.head()

zonas observadas: 11,633


,zona_id,celda_x,celda_y,n_hist,departamento,centro_x,centro_y
0,0,367,6285,1,SORIANO,367500,6285500
1,1,368,6249,4,COLONIA,368500,6249500
2,2,368,6250,15,COLONIA,368500,6250500
3,3,368,6251,2,COLONIA,368500,6251500
4,4,368,6258,2,SORIANO,368500,6258500


## 2. Agregación diaria por zona (sólo combinaciones observadas)

In [5]:
observado = df.groupby(["Fecha", "zona_id"]).size().rename("n_siniestros").reset_index()

fechas = pd.date_range(df["Fecha"].min(), df["Fecha"].max(), freq="D")
print(f"rango: {fechas.min():%Y-%m-%d} a {fechas.max():%Y-%m-%d}  ({len(fechas):,} días)")
print(f"pares (día, zona) con al menos un siniestro: {len(observado):,}")
print(f"máximo de siniestros en un mismo día y zona: {observado['n_siniestros'].max()}")

rango: 2018-01-01 a 2025-12-31  (2,922 días)
pares (día, zona) con al menos un siniestro: 209,391
máximo de siniestros en un mismo día y zona: 11


## 3. Panel completo día × zona, relleno con ceros

El producto cartesiano son ~54 M de filas: materializarlo entero en un `DataFrame` no entra en
memoria. Se construye primero la **matriz densa** `días × zonas` en `int16` (~108 MB) y desde ahí
se escribe el formato largo a Parquet **por bloques de zonas**, sin picos de memoria.

In [6]:
matriz = np.zeros((len(fechas), len(zonas)), dtype="int16")
filas = fechas.get_indexer(observado["Fecha"])
columnas = observado["zona_id"].to_numpy()
matriz[filas, columnas] = observado["n_siniestros"].to_numpy()

assert matriz.sum() == len(df), "se perdieron siniestros al armar el panel"
print(f"matriz {matriz.shape} · {matriz.nbytes / 1e6:.0f} MB · total siniestros {matriz.sum():,}")

matriz (2922, 11633) · 68 MB · total siniestros 224,694


In [7]:
DIR_SALIDA = RAIZ / "data" / "processed"
DIR_SALIDA.mkdir(parents=True, exist_ok=True)
RUTA_PANEL = DIR_SALIDA / "panel_diario_zona.parquet"
RUTA_ZONAS = DIR_SALIDA / "zonas.parquet"

zonas.to_parquet(RUTA_ZONAS, index=False)

esquema = pa.schema([("zona_id", pa.int32()), ("fecha", pa.date32()), ("n_siniestros", pa.int16())])
zona_ids = zonas["zona_id"].to_numpy()
fechas_np = fechas.to_numpy()
BLOQUE = 1000  # zonas por bloque

# Ordenado por (zona_id, fecha): cada serie temporal queda contigua y el Parquet comprime mucho mejor
with pq.ParquetWriter(RUTA_PANEL, esquema, compression="zstd") as escritor:
    for inicio in range(0, len(zonas), BLOQUE):
        corte = slice(inicio, inicio + BLOQUE)
        ids = zona_ids[corte]
        bloque = pd.DataFrame(
            {
                "zona_id": np.repeat(ids, len(fechas)),
                "fecha": np.tile(fechas_np, len(ids)),
                "n_siniestros": matriz[:, corte].T.ravel(),
            }
        )
        escritor.write_table(pa.Table.from_pandas(bloque, schema=esquema, preserve_index=False))

print(f"{RUTA_PANEL.name}: {RUTA_PANEL.stat().st_size / 1e6:.1f} MB")
print(f"{RUTA_ZONAS.name}: {RUTA_ZONAS.stat().st_size / 1e3:.0f} kB")

panel_diario_zona.parquet: 9.0 MB
zonas.parquet: 129 kB


## 4. Verificación

In [8]:
meta = pq.ParquetFile(RUTA_PANEL).metadata
esperadas = len(fechas) * len(zonas)
assert meta.num_rows == esperadas, (meta.num_rows, esperadas)

densidad = len(observado) / esperadas
print(f"filas del panel: {meta.num_rows:,} ({len(fechas):,} días × {len(zonas):,} zonas)")
print(f"filas distintas de cero: {len(observado):,} ({densidad:.3%})")
print(f"ceros: {1 - densidad:.3%}")

# Serie completa de la zona con más siniestros históricos, leída sin cargar todo el panel
zona_top = int(zonas.loc[zonas["n_hist"].idxmax(), "zona_id"])
serie = pq.read_table(RUTA_PANEL, filters=[("zona_id", "=", zona_top)]).to_pandas()
print(
    f"\nzona {zona_top} ({zonas.loc[zona_top, 'departamento']}): "
    f"{len(serie):,} filas · {serie['n_siniestros'].sum():,} siniestros"
)
serie.head()

filas del panel: 33,991,626 (2,922 días × 11,633 zonas)
filas distintas de cero: 209,391 (0.616%)
ceros: 99.384%

zona 9137 (MALDONADO): 2,922 filas · 2,030 siniestros


,zona_id,fecha,n_siniestros
0,9137,2018-01-01,0
1,9137,2018-01-02,3
2,9137,2018-01-03,1
3,9137,2018-01-04,0
4,9137,2018-01-05,1


In [9]:
# Cuántas zonas concentran los siniestros (útil para decidir un umbral mínimo de soporte)
acumulado = zonas["n_hist"].sort_values(ascending=False).cumsum() / len(df)
for pct in (0.5, 0.8, 0.9, 0.99):
    print(f"{pct:.0%} de los siniestros están en {(acumulado < pct).sum() + 1:,} zonas")

print("\nsiniestros históricos por zona:")
zonas["n_hist"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).round(1)

50% de los siniestros están en 221 zonas
80% de los siniestros están en 890 zonas
90% de los siniestros están en 2,120 zonas
99% de los siniestros están en 9,387 zonas

siniestros históricos por zona:


count    11633.0
mean        19.3
std         83.6
min          1.0
25%          1.0
50%          2.0
75%          6.0
90%         25.0
99%        397.4
max       2030.0
Name: n_hist, dtype: float64

## 5. Variables predictoras (features)

Todas las features son **causales**: la fila del día `t` sólo usa siniestros ocurridos hasta `t-1`.
Si una feature mirara el día `t`, el modelo vería su propia respuesta y la evaluación sería inválida.

| Feature | Definición |
|---|---|
| `lag_1`, `lag_7`, `lag_28` | siniestros en la misma zona hace 1 / 7 / 28 días |
| `media_7`, `media_28`, `media_365` | media móvil de siniestros de la zona en la ventana que **termina en `t-1`** |
| `dias_desde_ultimo` | días transcurridos desde el último siniestro de la zona, anterior a `t` |

Las variables de calendario (`dow`, `mes`, `dia_anio`, `es_feriado`) **no se almacenan**: son función
pura de `fecha` y se derivan al cargar, lo que ahorra 4 columnas sobre 47 M de filas.

Se descarta el primer año (2018) como **período de calentamiento**: `media_365` no está completa antes
de 2019-01-01.

> `n_hist` de `zonas.parquet` está calculado sobre los 8 años completos, **test incluido**. Es útil para
> describir, pero **no debe usarse como feature**: filtra información del futuro. Su equivalente
> causal es `media_365`.


In [10]:
SEMILLA = 42
LAGS = (1, 7, 28)
VENTANAS = (7, 28, 365)
ANIO_INICIO = 2019  # 2018 es calentamiento: media_365 incompleta

# Cortes del split temporal. El test queda reservado: sólo se evalúa sobre él la solución final.
ANIOS_TRAIN = range(ANIO_INICIO, 2024)  # 2019-2023
ANIOS_VALID = (2024,)
ANIOS_TEST = (2025,)


def features_bloque(sub: np.ndarray) -> dict[str, np.ndarray]:
    """Features causales para un bloque de zonas.

    `sub` es la submatriz densa (días × zonas) de conteos. Todas las salidas tienen su
    misma forma y la fila `t` sólo depende de `sub[:t]`.
    """
    n_dias = sub.shape[0]
    dias = np.arange(n_dias)
    feats: dict[str, np.ndarray] = {}

    for k in LAGS:
        rezago = np.zeros_like(sub)
        rezago[k:] = sub[:-k]
        feats[f"lag_{k}"] = rezago

    # Suma acumulada con una fila de ceros al inicio: acum[t] = suma de sub[0:t] (excluye t)
    acum = np.zeros((n_dias + 1, sub.shape[1]), dtype="float32")
    np.cumsum(sub, axis=0, out=acum[1:], dtype="float32")
    for w in VENTANAS:
        desde = np.maximum(dias - w, 0)
        feats[f"media_{w}"] = ((acum[dias] - acum[desde]) / w).astype("float32")

    # Último día con siniestro anterior a t
    marca = np.where(sub > 0, dias[:, None], -1)
    ultimo = np.maximum.accumulate(marca, axis=0)
    ultimo_previo = np.empty_like(ultimo)
    ultimo_previo[0] = -1
    ultimo_previo[1:] = ultimo[:-1]
    feats["dias_desde_ultimo"] = (dias[:, None] - ultimo_previo).astype("int16")

    return feats


In [11]:
import shutil

DIR_FEATURES = DIR_SALIDA / "features"
if DIR_FEATURES.exists():
    shutil.rmtree(DIR_FEATURES)

esquema_feat = pa.schema(
    [("zona_id", pa.int32()), ("fecha", pa.date32()), ("n_siniestros", pa.int8())]
    + [(f"lag_{k}", pa.int8()) for k in LAGS]
    + [(f"media_{w}", pa.float32()) for w in VENTANAS]
    + [("dias_desde_ultimo", pa.int16())]
)

# Particionado por año (estilo Hive): el split temporal se vuelve un filtro de directorio,
# sin leer los años que no corresponden.
anios = [a for a in sorted(fechas.year.unique()) if a >= ANIO_INICIO]
indices_anio = {a: np.where(fechas.year == a)[0] for a in anios}
escritores = {}

try:
    for inicio in range(0, len(zonas), BLOQUE):
        corte = slice(inicio, min(inicio + BLOQUE, len(zonas)))
        sub = matriz[:, corte]
        ids = zona_ids[corte]
        feats = features_bloque(sub)

        for anio in anios:
            sel = indices_anio[anio]
            tabla = {
                "zona_id": np.repeat(ids, len(sel)),
                "fecha": np.tile(fechas[sel].to_numpy(), len(ids)),
                "n_siniestros": sub[sel].T.ravel().astype("int8"),
            }
            for nombre, arreglo in feats.items():
                tabla[nombre] = arreglo[sel].T.ravel()

            if anio not in escritores:
                destino = DIR_FEATURES / f"anio={anio}"
                destino.mkdir(parents=True, exist_ok=True)
                escritores[anio] = pq.ParquetWriter(
                    destino / "datos.parquet", esquema_feat, compression="zstd"
                )
            escritores[anio].write_table(pa.Table.from_pydict(tabla, schema=esquema_feat))
finally:
    for escritor in escritores.values():
        escritor.close()

total = sum(f.stat().st_size for f in DIR_FEATURES.rglob("*.parquet"))
print(f"features: {total / 1e6:.0f} MB en {len(anios)} particiones")
for anio in anios:
    ruta = DIR_FEATURES / f"anio={anio}" / "datos.parquet"
    print(f"  {anio}: {pq.ParquetFile(ruta).metadata.num_rows:>10,} filas · {ruta.stat().st_size / 1e6:4.1f} MB")


features: 24 MB en 7 particiones
  2019:  4,246,045 filas ·  2.5 MB
  2020:  4,257,678 filas ·  3.2 MB
  2021:  4,246,045 filas ·  4.3 MB
  2022:  4,246,045 filas ·  4.7 MB
  2023:  4,246,045 filas ·  3.1 MB
  2024:  4,257,678 filas ·  3.0 MB
  2025:  4,246,045 filas ·  3.5 MB


In [12]:
# Verificación: las features tienen que coincidir exactamente con shift/rolling de pandas
zona_ctrl = zona_top
ctrl = (
    ds.dataset(DIR_FEATURES, format="parquet", partitioning="hive")
    .to_table(filter=ds.field("zona_id") == zona_ctrl)
    .to_pandas()
    .sort_values("fecha")
    .reset_index(drop=True)
)
ctrl["fecha"] = pd.to_datetime(ctrl["fecha"])

crudo = pq.read_table(RUTA_PANEL, filters=[("zona_id", "=", zona_ctrl)]).to_pandas()
crudo["fecha"] = pd.to_datetime(crudo["fecha"])
crudo = crudo.sort_values("fecha").set_index("fecha")["n_siniestros"]

for k in LAGS:
    esperado = crudo.shift(k).reindex(ctrl["fecha"]).to_numpy()
    obtenido = ctrl[f"lag_{k}"].to_numpy()
    valido = ~np.isnan(esperado)
    assert np.array_equal(esperado[valido], obtenido[valido]), f"lag_{k}"

for w in VENTANAS:
    # shift(1) antes del rolling: la ventana termina en t-1
    esperado = crudo.shift(1).rolling(w, min_periods=w).mean().reindex(ctrl["fecha"]).to_numpy()
    obtenido = ctrl[f"media_{w}"].to_numpy()
    valido = ~np.isnan(esperado)
    assert np.allclose(esperado[valido], obtenido[valido], atol=1e-6), f"media_{w}"

assert ctrl["n_siniestros"].sum() == crudo.loc[f"{ANIO_INICIO}":].sum()
print(f"zona {zona_ctrl}: lags y medias móviles coinciden con shift/rolling de pandas ✓")
print(f"{len(ctrl):,} filas · {ctrl['n_siniestros'].sum():,} siniestros desde {ANIO_INICIO}")


zona 9137: lags y medias móviles coinciden con shift/rolling de pandas ✓
2,557 filas · 1,748 siniestros desde 2019


## 6. Split temporal y muestra de entrenamiento

El split es **por fecha**, nunca aleatorio: con features de rezago, un split aleatorio le filtra el
futuro al modelo y las métricas salen infladas.

| Partición | Años | Filas |
|---|---|---|
| Entrenamiento | 2019–2023 | 33,8 M |
| Validación | 2024 | 6,8 M |
| Test (reservado) | 2025 | 6,7 M |

**Submuestreo de ceros.** El train completo son 1,1 GB en pandas, más de lo que hay disponible en
esta máquina. Como el 99,6% de las filas son ceros, se conservan **todos los positivos** y una
fracción `P_CERO` de los ceros, compensando con un **peso `1/P_CERO`** en los ceros conservados.
Es el esquema *case-control* estándar para eventos raros: con objetivo Poisson, el valor esperado
que predice el modelo sigue calibrado (se verifica abajo comparando la media ponderada contra la real).

El test **no se submuestrea**: se evalúa sobre el panel completo.


In [13]:
COLS_FEATURES = (
    [f"lag_{k}" for k in LAGS]
    + [f"media_{w}" for w in VENTANAS]
    + ["dias_desde_ultimo", "dow", "mes", "dia_anio", "es_feriado"]
)
FERIADOS_UY = holidays.country_holidays("UY", years=range(ANIO_INICIO, 2026))


def agregar_calendario(datos: pd.DataFrame) -> pd.DataFrame:
    """Variables de calendario derivadas de `fecha` (no se almacenan en disco)."""
    datos["dow"] = datos["fecha"].dt.dayofweek.astype("int8")
    datos["mes"] = datos["fecha"].dt.month.astype("int8")
    datos["dia_anio"] = datos["fecha"].dt.dayofyear.astype("int16")
    datos["es_feriado"] = datos["fecha"].dt.date.isin(FERIADOS_UY).astype("int8")
    return datos


def cargar_particion(anios, p_cero: float | None = None, semilla: int = SEMILLA) -> pd.DataFrame:
    """Carga los años pedidos. Si `p_cero` se indica, submuestrea los ceros y agrega `peso`.

    Lee año por año y submuestrea antes de concatenar, para no materializar la partición
    entera en memoria.
    """
    conjunto = ds.dataset(DIR_FEATURES, format="parquet", partitioning="hive")
    rng = np.random.default_rng(semilla)
    partes = []

    for anio in anios:
        datos = conjunto.to_table(filter=ds.field("anio") == anio).to_pandas()
        if p_cero is not None:
            positivo = datos["n_siniestros"].to_numpy() > 0
            datos = datos[positivo | (rng.random(len(datos)) < p_cero)]
        partes.append(datos)

    datos = pd.concat(partes, ignore_index=True)
    datos["fecha"] = pd.to_datetime(datos["fecha"])
    datos = agregar_calendario(datos)
    datos["peso"] = (
        1.0
        if p_cero is None
        else np.where(datos["n_siniestros"].to_numpy() > 0, 1.0, 1.0 / p_cero)
    )
    return datos


P_CERO = 0.08
train = cargar_particion(ANIOS_TRAIN, p_cero=P_CERO)
valid = cargar_particion(ANIOS_VALID, p_cero=P_CERO)

print(f"train: {len(train):,} filas · {train.memory_usage(deep=True).sum() / 1e6:.0f} MB")
print(f"valid: {len(valid):,} filas · {valid.memory_usage(deep=True).sum() / 1e6:.0f} MB")
print(f"positivos en train: {(train['n_siniestros'] > 0).sum():,}")


train: 1,813,447 filas · 85 MB
valid: 366,105 filas · 17 MB
positivos en train: 125,933


In [14]:
# Verificación 1: no hay solapamiento temporal entre particiones
assert train["fecha"].max() < valid["fecha"].min(), "train y validación se solapan"
print(f"train  {train['fecha'].min():%Y-%m-%d} → {train['fecha'].max():%Y-%m-%d}")
print(f"valid  {valid['fecha'].min():%Y-%m-%d} → {valid['fecha'].max():%Y-%m-%d}")
print(f"test   reservado: {ANIOS_TEST[0]} (no se carga hasta la evaluación final)")

# Verificación 2: el submuestreo con pesos no sesga el valor esperado
conjunto = ds.dataset(DIR_FEATURES, format="parquet", partitioning="hive")
media_real = (
    conjunto.to_table(filter=ds.field("anio").isin(list(ANIOS_TRAIN)), columns=["n_siniestros"])
    .column("n_siniestros")
    .to_numpy()
    .mean()
)
media_ponderada = np.average(train["n_siniestros"], weights=train["peso"])
print(f"\nmedia real de siniestros/día/zona en train: {media_real:.6f}")
print(f"media ponderada de la muestra:              {media_ponderada:.6f}")
assert abs(media_ponderada - media_real) / media_real < 0.02, "el submuestreo sesga la media"
print("el peso 1/P_CERO conserva el valor esperado ✓")


train  2019-01-01 → 2023-12-31
valid  2024-01-01 → 2024-12-31
test   reservado: 2025 (no se carga hasta la evaluación final)

media real de siniestros/día/zona en train: 0.006349
media ponderada de la muestra:              0.006355
el peso 1/P_CERO conserva el valor esperado ✓
